# 1. Sentiment Classification

In [ ]:
# Install required packages
# !pip install numpy pandas nltk scikit-learn matplotlib seaborn wordcloud

# Download NLTK data
import nltk

nltk.download("movie_reviews")
nltk.download("stopwords")
nltk.download("punkt")
nltk.download("wordnet")
nltk.download("averaged_perceptron_tagger")
nltk.download("punkt_tab")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Set random seed for reproducibility
np.random.seed(42)

In [ ]:
# Load movie reviews dataset

from nltk.corpus import movie_reviews

file_ids = movie_reviews.fileids()
movie_reviews_dict = {"label": [], "text": [], "length": []}
for file_id in file_ids:
    movie_reviews_dict["label"].append(file_id)
    movie_reviews_dict["text"].append(movie_reviews.raw(file_id))
    movie_reviews_dict["length"].append(len(movie_reviews.raw(file_id)))

df = pd.DataFrame.from_dict(movie_reviews_dict)
df = df.sample(frac=1).reset_index(drop=True)

# Map labels to sentiment (pos/neg)
df["sentiment"] = df["label"].apply(lambda x: "positive" if "pos" in x else "negative")

print("---------------- INFO ----------------")
display(df.info())
print("---------------- DESCRIBE ----------------")
display(df.describe())

In [ ]:
# Text Preprocessing

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer, PorterStemmer
import re
import string

stop_words = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()
stemmer = PorterStemmer()


def preprocess_text(text):
    """
    Advanced preprocessing: + lemmatization, remove numbers/special chars

    Args:
        text (str): Input text
    Returns:
        str: Preprocessed text (space-separated tokens)
    """
    # 1. Lowercase
    text = text.lower()

    # 2. Remove numbers and special characters (keep only letters and spaces)
    text = re.sub(r"[^a-z\s]", "", text)

    # 3. Tokenize
    tokens = word_tokenize(text)

    # 4. Remove stopwords
    tokens = [t for t in tokens if t not in stop_words]

    # 5. Lemmatize
    tokens = [lemmatizer.lemmatize(t) for t in tokens]

    return " ".join(tokens)

In [ ]:
# Test preprocessing function
sample_text = "This movie is AMAZING!!! I've watched it 3 times. Best film of 2023! :)"

print("Original:", sample_text)
print("Preprocessed:", preprocess_text(sample_text))

In [ ]:
# Applying advanced preprocessing
df["processed_text"] = df["text"].apply(preprocess_text)
print("\nDataFrame after applying preprocessing:")
display(df.head())

In [ ]:
# Split data into train/test sets
from sklearn.model_selection import train_test_split

X = df["processed_text"]
y = df["sentiment"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("\nTrain/Test Split:")
print(f"\nX_train shape: {X_train.shape}")
print(f"\nX_test shape: {X_test.shape}")
print(f"\ny_train distribution:\n{y_train.value_counts()}")
print(f"\ny_test distribution:\n{y_test.value_counts()}")

### Extracting Features

In [ ]:
# Feature Extraction Method 1 - Bag of Words
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, classification_report


# Encode labels
le = LabelEncoder()
y_train_enc = le.fit_transform(y_train)
y_test_enc = le.transform(y_test)

pipeline = Pipeline(
    [("vectorizer", CountVectorizer()), ("clf", LogisticRegression(max_iter=1000))]
)

param_grid = {
    "vectorizer__max_features": [2000, 5000, 8000],
    "vectorizer__min_df": [1, 2],
    "vectorizer__max_df": [0.95, 0.9],
}

grid = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    scoring="accuracy",
    cv=5,
    n_jobs=-1,
    verbose=1,
)

grid.fit(X_train, y_train_enc)

print("Bag of Words:")
print("Best parameters:")
print(grid.best_params_)

print("\nBest CV accuracy:")
print(grid.best_score_)

best_model = grid.best_estimator_

y_pred = best_model.predict(X_test)

print("\nTest accuracy:")
print(accuracy_score(y_test_enc, y_pred))

print("\nClassification report:")
print(classification_report(y_test_enc, y_pred, target_names=le.classes_))

best_vectorizer = best_model.named_steps["vectorizer"]
print("Best vocabulary size:", len(best_vectorizer.get_feature_names_out()))

bow_vectorizer = CountVectorizer(max_features=8000, min_df=2, max_df=0.99)

X_train_bow = bow_vectorizer.fit_transform(X_train)
X_test_bow = bow_vectorizer.transform(X_test)

In [ ]:
# Feature Extraction Method 2 - TF-IDF
from sklearn.feature_extraction.text import TfidfVectorizer

pipeline = Pipeline(
    [("vectorizer", TfidfVectorizer()), ("clf", LogisticRegression(max_iter=1000))]
)


grid = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    scoring="accuracy",
    cv=5,
    n_jobs=-1,
    verbose=1,
)

grid.fit(X_train, y_train_enc)

print("TF-IDF:")
print("Best parameters:")
print(grid.best_params_)

print("\nBest CV accuracy:")
print(grid.best_score_)

best_model = grid.best_estimator_

y_pred = best_model.predict(X_test)

print("\nTest accuracy:")
print(accuracy_score(y_test_enc, y_pred))

print("\nClassification report:")
print(classification_report(y_test_enc, y_pred, target_names=le.classes_))

best_vectorizer = best_model.named_steps["vectorizer"]
print("Best vocabulary size:", len(best_vectorizer.get_feature_names_out()))

tfidf_vectorizer = TfidfVectorizer(max_features=2000, min_df=2, max_df=0.9)

X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
X_test_tfidf = tfidf_vectorizer.transform(X_test)

### Find Best Model

In [ ]:
from sklearn.naive_bayes import MultinomialNB, BernoulliNB
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import classification_report, confusion_matrix
import time

def train_and_evaluate(
    model, X_train, X_test, y_train, y_test, model_name, feature_name
):
    """
    Train a model and return evaluation metrics

    Args:
        model: sklearn classifier
        X_train, X_test: feature matrices
        y_train, y_test: labels
        model_name: name of the model (str)
        feature_name: name of feature type (str)

    Returns:
        dict: Dictionary containing all metrics
    """

    # -------------------- Training --------------------
    start_time = time.time()
    model.fit(X_train, y_train)
    train_time = time.time() - start_time

    # -------------------- Prediction --------------------
    y_pred = model.predict(X_test)

    # -------------------- Metrics --------------------
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, average="weighted", zero_division=0)
    recall = recall_score(y_test, y_pred, average="weighted", zero_division=0)
    f1 = f1_score(y_test, y_pred, average="weighted", zero_division=0)

    # -------------------- Results --------------------
    return {
        "Model": model_name,
        "Feature_Type": feature_name,
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1_Score": f1,
        "Training_Time_sec": train_time,
    }

In [ ]:
# Train all models on all feature types

results = []

models = {
    "Multinomial NB": MultinomialNB(),
    "Bernoulli NB": BernoulliNB(),
    "Logistic Regression": LogisticRegression(max_iter=1000),
}

feature_sets = {
    "Bag-of-Words": (X_train_bow, X_test_bow),
    "TF-IDF": (X_train_tfidf, X_test_tfidf),
}

# Loop through all combinations of models and features
# Use train_and_evaluate function
# Append results to list

# Convert results to DataFrame for easy analysis
# results_df = pd.DataFrame(results)

# Train all models on all feature types
results = []

for feature_name, (Xtr, Xte) in feature_sets.items():
    for model_name, model in models.items():
        result = train_and_evaluate(
            model=model,
            X_train=Xtr,
            X_test=Xte,
            y_train=y_train_enc,
            y_test=y_test_enc,
            model_name=model_name,
            feature_name=feature_name,
        )
        results.append(result)

# Convert results to DataFrame for easy analysis
results_df = pd.DataFrame(results)

In [ ]:
# Display comprehensive results table
results_df = results_df.sort_values(by=["F1_Score", "Accuracy"], ascending=False)

display(results_df)

In [ ]:
# TODO: Find best models
best_overall = results_df.sort_values("F1_Score", ascending=False).iloc[0]

print("Best overall model:")
display(best_overall)

### Confusion Matrix

In [ ]:
# Display confusion matrix for best model

from sklearn.metrics import ConfusionMatrixDisplay

best_model_name = best_overall["Model"]
best_feature_name = best_overall["Feature_Type"]

# Retrieve best model and features
best_model = models[best_model_name]
Xtr_best, Xte_best = feature_sets[best_feature_name]

# Retrain best model on full training data
best_model.fit(Xtr_best, y_train_enc)
y_pred_best = best_model.predict(Xte_best)

# Confusion matrix
cm = confusion_matrix(y_test_enc, y_pred_best)

disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=le.classes_)
disp.plot()
plt.title(f"Confusion Matrix: {best_model_name} + {best_feature_name}")
plt.show()

### Error Analysis

In [ ]:
# Labelling data
test_df = df.loc[X_test.index].copy()

test_df["y_true"] = y_test_enc
test_df["y_pred"] = y_pred_best

test_df["true_label"] = le.inverse_transform(test_df["y_true"])
test_df["pred_label"] = le.inverse_transform(test_df["y_pred"])

test_df["correct"] = test_df["y_true"] == test_df["y_pred"]

display(test_df.head())

In [ ]:
# Find and categorize misclassifications
false_positives = test_df[
    (test_df["pred_label"] == "positive") & (test_df["true_label"] == "negative")
]

false_negatives = test_df[
    (test_df["pred_label"] == "negative") & (test_df["true_label"] == "positive")
]

print("False Positives:", len(false_positives))
display(false_positives.head())
print("False Negatives:", len(false_negatives))
display(false_negatives.head())

In [ ]:
# Analyze error patterns
print("Average length (correct):", test_df[test_df["correct"]]["length"].mean())

print("Average length (incorrect):", test_df[~test_df["correct"]]["length"].mean())

from collections import Counter

all_misclassified_text = " ".join(test_df[~test_df["correct"]]["processed_text"])
word_counts = Counter(all_misclassified_text.split())

print("Most common words in misclassified reviews:")
display(word_counts.most_common(10))

In [ ]:
# Visualization - Length distribution
import matplotlib.pyplot as plt

plt.figure()
plt.hist(test_df[test_df["correct"]]["length"], bins=30, alpha=0.7, label="Correct")
plt.hist(test_df[~test_df["correct"]]["length"], bins=30, alpha=0.7, label="Incorrect")

plt.xlabel("Review Length")
plt.ylabel("Frequency")
plt.title("Review Length Distribution: Correct vs Incorrect")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Visualization - Word cloud of misclassified reviews
from wordcloud import WordCloud

wordcloud = WordCloud(width=800, height=400, background_color="white").generate(
    all_misclassified_text
)

plt.figure(figsize=(10, 5))
plt.imshow(wordcloud, interpolation="bilinear")
plt.axis("off")
plt.title("Word Cloud of Misclassified Reviews")
plt.show()

# 2: Word Vectors & Similarity

In [ ]:
unique_text = set(" ".join(df["processed_text"]).split())

sorted_unique_text = sorted(list(unique_text))
word2id = {word: idx for idx, word in enumerate(sorted_unique_text)}
word2id

In [ ]:
co_occurrence_matrix = np.zeros(
    (len(sorted_unique_text), len(sorted_unique_text)), dtype=int
)
co_occurrence_matrix.shape

In [ ]:
window_size = 3

# loop through each document
for text in df["processed_text"]:
    tokens = text.split()
    token_ids = [word2id[t] for t in tokens if t in word2id]

    # loop through each token in the document
    for i, token_id in enumerate(token_ids):
        start = max(0, i - window_size)
        end = min(len(token_ids), i + window_size + 1)

        for j in range(start, end):
            ## avoid self-co-occurrence
            if i != j:
                co_occurrence_matrix[token_id, token_ids[j]] += 1

co_occurrence_matrix

### Cosine Similarity

In [ ]:
## 2: Word Vectors & Similarity
def get_cosine_similarity(vec1, vec2):
    """
    Compute cosine similarity between two vectors

    Args:
        vec1, vec2: numpy arrays

    Returns:
        float: cosine similarity
    """
    dot_product = np.dot(vec1, vec2)
    norm_vec1 = np.linalg.norm(vec1)
    norm_vec2 = np.linalg.norm(vec2)

    if norm_vec1 == 0 or norm_vec2 == 0:
        return 0.0

    return dot_product / (norm_vec1 * norm_vec2)

In [ ]:
# test compare 2 word vectors
word1 = "film"
word2 = "movie"

vec1 = co_occurrence_matrix[word2id[word1]]
vec2 = co_occurrence_matrix[word2id[word2]]

similarity = get_cosine_similarity(vec1, vec2)
similarity

In [ ]:
def find_top_similar_words(target_word, top_k=5):
    """
    Find top K similar words to the target word based on cosine similarity

    Args:
        target_word (str): The word to find similarities for
        top_k (int): Number of top similar words to return

    Returns:
        list of tuples: List of (word, similarity) tuples
    """
    if target_word not in word2id:
        return []

    target_vec = co_occurrence_matrix[word2id[target_word]]
    similarities = []

    for word, idx in word2id.items():
        if word != target_word:
            vec = co_occurrence_matrix[idx]
            sim = get_cosine_similarity(target_vec, vec)
            similarities.append((word, sim))

    # Sort by similarity and return top K
    similarities.sort(key=lambda x: x[1], reverse=True)
    return similarities[:top_k]

In [ ]:
find_top_similar_words("good")

### PPMI (Positive Pointwise Mutual Information)
Indicating how much more often two events co-occur than if they were independent

In [ ]:
# PPMI: calculate joint and marginal probabilities
total_count = np.sum(co_occurrence_matrix)
# P( X, Y ): x and y co-occurrence
joint_probs = co_occurrence_matrix / total_count  

marginal_probs = np.sum(joint_probs, axis=1)
# P( X ) * P( Y ): x and y independent
expected_probs = np.outer(marginal_probs, marginal_probs)

In [ ]:
pmi = np.zeros_like(joint_probs)
non_zero_indices = joint_probs > 0
pmi[non_zero_indices] = np.log2(
    joint_probs[non_zero_indices] / expected_probs[non_zero_indices]
)
ppmi = np.maximum(pmi, 0)

In [ ]:
ppmi


### PCA

In [ ]:
from sklearn.decomposition import PCA

pca = PCA(n_components=2)
word_vectors_2d = pca.fit_transform(ppmi)

In [ ]:
plt.figure(figsize=(12, 12))

plt.scatter(word_vectors_2d[:, 0], word_vectors_2d[:, 1], alpha=0.5)
for word, index in word2id.items():
    x = word_vectors_2d[index, 0]
    y = word_vectors_2d[index, 1]

    plt.text(x, y, word, fontsize=9)

plt.show()

### Word analogy

In [ ]:
# write word analogy function
def word_analogy(word_a, word_b, word_c, word_vector, top_k=5):
    """
    Solve word analogy: word_a - word_b + word_c = ?

    Args:
        word_a, word_b, word_c (str): Input words
        top_k (int): Number of top similar words to return

    Returns:
        list of tuples: List of (word, similarity) tuples
    """
    if word_a not in word2id or word_b not in word2id or word_c not in word2id:
        return []

    vec_a = word_vector[word2id[word_a]]
    vec_b = word_vector[word2id[word_b]]
    vec_c = word_vector[word2id[word_c]]

    target_vec = vec_a - vec_b + vec_c

    similarities = []

    for word, idx in word2id.items():
        if word not in [word_a, word_b, word_c]:
            vec = word_vector[idx]
            sim = get_cosine_similarity(target_vec, vec)
            similarities.append((word, sim))

    # Sort by similarity and return top K
    similarities.sort(key=lambda x: x[1], reverse=True)
    return similarities[:top_k]

In [ ]:
word_analogy("actor", "man", "woman", ppmi)

# 3. Document Search

In [ ]:
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# 1. Dữ liệu giả lập (Corpus)
documents = [
    "I love machine learning and deep learning",  # Doc 0
    "Deep learning is a subset of machine learning",  # Doc 1
    "I prefer watching movies over reading books",  # Doc 2
    "Machine learning is fascinating",  # Doc 3
]

# 2. Xây dựng TF-IDF vectors
# TfidfVectorizer sẽ tự động thực hiện: Tokenize -> Đếm từ -> Tính TF-IDF
vectorizer = TfidfVectorizer(
    stop_words="english"
)  # stop_words loại bỏ các từ như 'is', 'a', 'the'
tfidf_matrix = vectorizer.fit_transform(documents)

# In ra kích thước ma trận để kiểm tra (số documents x số từ vựng)
print(f"Kích thước ma trận TF-IDF: {tfidf_matrix.shape}")
print(f"Các từ vựng (features): {vectorizer.get_feature_names_out()}\n")


# 3. & 4. Hàm tìm kiếm và Xếp hạng
def search_documents(query, top_k=2):
    """
    Tìm kiếm văn bản dựa trên query sử dụng Cosine Similarity
    """
    # Bước A: Biến đổi query thành vector (sử dụng vocab đã học từ documents)
    query_vec = vectorizer.transform([query])

    # Bước B: Tính độ tương đồng giữa query và tất cả documents
    # Kết quả là mảng (1, n_docs) chứa điểm số similarity
    similarities = cosine_similarity(query_vec, tfidf_matrix).flatten()

    # Bước C: Sắp xếp kết quả giảm dần (từ tương đồng nhất đến ít nhất)
    # argsort trả về index, [::-1] để đảo ngược mảng thành giảm dần
    top_indices = similarities.argsort()[::-1][:top_k]

    # Bước D: Trả về kết quả
    results = []
    for idx in top_indices:
        score = similarities[idx]
        if score > 0:  # Chỉ lấy các kết quả có liên quan
            results.append((documents[idx], score))

    return results


# 5. Đánh giá với Query mẫu (Simple Interface)
queries = ["machine learning", "deep learning", "watching movies", "reading"]

print("--- KẾT QUẢ TÌM KIẾM ---")
for query in queries:
    print(f"\nQuery: '{query}'")
    results = search_documents(query)

    if not results:
        print(" -> Không tìm thấy kết quả phù hợp.")
    else:
        for doc, score in results:
            print(f" -> [Score: {score:.4f}] {doc}")